# Ancestry clusters and disease-specific CV

This notebook creates ancestry clusters from PC1-PC10, displays the PC1-PC2 cluster plot, and generates disease-specific 5-fold splits.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

from decipehr.ancestry import (
    DEFAULT_PC_COLS,
    ancestry_cluster_report,
    cluster_quality_report,
    make_ancestry_clusters,
    write_pc_scatter_svg,
)
from decipehr.eligibility import load_eligibility_rules
from decipehr.io import read_table, write_table
from decipehr.splits import disease_fold_balance_report, make_disease_specific_folds


ModuleNotFoundError: No module named 'decipehr'

In [ ]:
phenotype_path = Path("/gpfs/data/deciphEHRlab/deciphEHRdream/training_phase/phenotype_files/Training_Cov_file.txt")
clusters_path = Path("data/processed/ancestry_clusters.csv")
cluster_report_path = Path("outputs/ancestry/ancestry_cluster_report.csv")
quality_path = Path("outputs/ancestry/ancestry_cluster_quality.csv")
plot_path = Path("outputs/ancestry/pc1_pc2_clusters.svg")
splits_path = Path("data/processed/splits/disease_specific_split_seed42.csv")
split_report_path = Path("outputs/cv/disease_specific_split_summary.csv")

patient_col = "sampleid"
labels = ["cad", "bc", "T2D"]
age_col = "age_sd"
sex_col = "sex"


In [ ]:
phenotype = read_table(phenotype_path)
phenotype[[patient_col, sex_col, age_col, *DEFAULT_PC_COLS, *labels]].head()


In [ ]:
quality = cluster_quality_report(phenotype, DEFAULT_PC_COLS, k_min=2, k_max=6, seed=42, n_init=50)
write_table(quality, quality_path)
quality


In [ ]:
clusters = make_ancestry_clusters(
    phenotype,
    patient_col=patient_col,
    pc_cols=DEFAULT_PC_COLS,
    n_clusters=3,
    seed=42,
    n_init=50,
)
write_table(clusters, clusters_path)

cluster_report = ancestry_cluster_report(
    phenotype,
    clusters,
    patient_col=patient_col,
    pc_cols=DEFAULT_PC_COLS,
    label_cols=labels,
    age_col=age_col,
    sex_col=sex_col,
)
write_table(cluster_report, cluster_report_path)
cluster_report


In [ ]:
write_pc_scatter_svg(
    phenotype,
    clusters,
    plot_path,
    patient_col=patient_col,
)
display(SVG(filename=str(plot_path)))


In [ ]:
phenotype_with_clusters = phenotype.merge(clusters, on=patient_col)
eligibility_rules = load_eligibility_rules("configs/disease_eligibility.yaml")

folds = make_disease_specific_folds(
    phenotype_with_clusters,
    patient_col=patient_col,
    label_cols=labels,
    cluster_col="ancestry_cluster",
    age_col=age_col,
    sex_col=sex_col,
    sex_balance_labels=["cad", "T2D"],
    eligibility_rules=eligibility_rules,
    n_folds=5,
    seed=42,
    n_seed_candidates=100,
)
write_table(folds, splits_path)
folds.head()


In [ ]:
split_summary = disease_fold_balance_report(
    phenotype_with_clusters,
    folds,
    patient_col,
    labels,
    cluster_col="ancestry_cluster",
    age_col=age_col,
    sex_col=sex_col,
)
write_table(split_summary, split_report_path)
split_summary
